In [14]:
import numpy as np
import re
import nltk
import joblib
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer


In [15]:
best_model = joblib.load("models/spam_detection_model.joblib")

In [16]:
# Preprocess function that mirrors the training-time preprocessing
def preprocess_message(message):
    stop_words = set(stopwords.words("english"))
    stemmer = PorterStemmer()
    message = message.lower()
    message = re.sub(r"[^a-z\s$!]", "", message)
    tokens = word_tokenize(message)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

In [17]:
# Example SMS messages for evaluation
new_messages = [
    # Likely spam
    "WINNER!! You have been selected for a free cruise to the Bahamas. Reply YES to claim your prize.",
    "Your package could not be delivered. Pay the $1.99 redelivery fee at www.parcel-track-example.com",
    "Get cheap meds online, no prescription needed! Visit our store today, 70% off.",
    "You have an unclaimed tax refund of $842. Click here to receive it: www.refund-example.net",
    "Hot singles in your area want to chat! Text CHAT to 69420 now.",
    "Final notice: your car warranty is about to expire. Call 1-800-555-0199 immediately.",

    # Likely ham
    "Can you pick up milk and eggs on your way home?",
    "Thanks for dinner last night, it was great to catch up!",
    "The meeting has been moved to 3pm in conference room B.",
    "Happy birthday! Hope you have an amazing day.",
    "I'm running about 10 minutes late, sorry!",

    # Tricky edge cases
    "Congrats on the new job! We should celebrate this weekend.",
    "Your verification code is 482913. Do not share it with anyone.",
    "Don't forget to claim your free coffee at the office kitchen today.",
    "Your bank statement for September is now available in the app.",
]


In [18]:
# Preprocess and vectorize messages
processed_messages = [preprocess_message(msg) for msg in new_messages]

# Transform preprocessed messages into feature vectors
X_new = best_model.named_steps["vectorizer"].transform(processed_messages)

In [19]:
# Predict with the trained classifier
predictions = best_model.named_steps["classifier"].predict(X_new)
prediction_probabilities = best_model.named_steps["classifier"].predict_proba(X_new)


In [20]:
# Display predictions and probabilities for each evaluated message
for i, msg in enumerate(new_messages):
    prediction = "Spam" if predictions[i] == 1 else "Not-Spam"
    spam_probability = prediction_probabilities[i][1]  # Probability of being spam
    ham_probability = prediction_probabilities[i][0]   # Probability of being not spam
    
    print(f"Message: {msg}")
    print(f"Prediction: {prediction}")
    print(f"Spam Probability: {spam_probability:.2f}")
    print(f"Not-Spam Probability: {ham_probability:.2f}")
    print("-" * 50)


Message: WINNER!! You have been selected for a free cruise to the Bahamas. Reply YES to claim your prize.
Prediction: Spam
Spam Probability: 1.00
Not-Spam Probability: 0.00
--------------------------------------------------
Message: Your package could not be delivered. Pay the $1.99 redelivery fee at www.parcel-track-example.com
Prediction: Not-Spam
Spam Probability: 0.00
Not-Spam Probability: 1.00
--------------------------------------------------
Message: Get cheap meds online, no prescription needed! Visit our store today, 70% off.
Prediction: Not-Spam
Spam Probability: 0.01
Not-Spam Probability: 0.99
--------------------------------------------------
Message: You have an unclaimed tax refund of $842. Click here to receive it: www.refund-example.net
Prediction: Spam
Spam Probability: 0.98
Not-Spam Probability: 0.02
--------------------------------------------------
Message: Hot singles in your area want to chat! Text CHAT to 69420 now.
Prediction: Spam
Spam Probability: 1.00
Not-Spa